# Módulo 1 · Etapa 6 — Aumento de datos
**TFM · HAM10000** · Responsable: **Adrián Arias** · MLflow: experimento `m1-datos`, run `m1-aumento`, tag `fase=aumento`

**Objetivo:** dar variedad al entrenamiento sin cambiar el diagnóstico. Pregunta guía: *¿un dermatólogo seguiría dando el mismo diagnóstico tras esta transformación?*

**Qué hace**
1. Escribe el módulo `src/augmentation.py`, que usará el pipeline de la Etapa 7. Se guarda también en Drive.
2. Fija el **catálogo**: qué transformaciones se usan, con qué rango y por qué; y cuáles se descartan.
3. **Aumento selectivo:** intensidad *fuerte* en todas las clases salvo `nv`; *ligera* (solo volteos) en `nv`.
4. Comprueba automáticamente que el aumento **solo se activa en entrenamiento**, que es reproducible y que no se sale del rango de píxeles.
5. Genera la figura de variantes por clase para **revisarla a ojo** y otra que justifica el relleno por reflejo.

**Parámetros:** los de `configs/base.yaml` (ya acordados por el grupo), más el relleno por reflejo, que se añade a partir de la Etapa 3.

**Dependencia:** antes del aumento, la imagen se recorta al centro (450×450) y se reduce a 224×224 por área.

Entorno: **CPU**, unos 3 minutos (no hace falta copiar el dataset: lee solo las imágenes de las figuras). Secretos: `DAGSHUB_USER`, `DAGSHUB_TOKEN`.

## 1. Preparación de la sesión

In [ ]:
AUTOR = "adrian"
OWNER, REPO = "aReinoso007", "tfm-melanoma-cnn-gradcam"
DATASET_SHA256_ESPERADO = "06aee5e603d313ba69de38263053c792a1dff030eb9c44a96ceb477ceefb99a5"
SPLIT_SHA256_ESPERADO = "98485423d758cae1ed8b7adf64c3e92d31a6afd4d48de7a26e89355238e34b38"
SEMILLA_FIG = 42             # cambiar este número para revisar otra tanda de imágenes (sección 6)
CAMBIOS_PENDIENTES_YAML = {}   # se mantienen los valores de base.yaml tras la revisión visual

In [ ]:
import os, json, zipfile, io, shutil
from pathlib import Path
from google.colab import drive, userdata
drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/TFM")

REPO_DIR = f"/content/{REPO}"
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/{OWNER}/{REPO}.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
%cd {REPO_DIR}
!pip install -q -r requirements.txt

In [ ]:
import numpy as np, pandas as pd, tensorflow as tf
from PIL import Image
from src.tracking import load_config, file_sha256
cfg = load_config("configs/base.yaml")
print("TensorFlow", tf.__version__)

SPLIT_DRIVE = DRIVE_ROOT / "data" / "splits" / "split_v1.csv"
assert file_sha256(SPLIT_DRIVE) == SPLIT_SHA256_ESPERADO, "[!] El split de Drive no es el split v1 publicado"
split = pd.read_csv(SPLIT_DRIVE)

# Las imágenes se leen directamente del zip de Drive (solo hacen falta unas pocas)
ZIP = zipfile.ZipFile(cfg["paths"]["dataset_zip"])
assert __import__("hashlib").sha256(ZIP.read("HAM10000_metadata.csv")).hexdigest() == DATASET_SHA256_ESPERADO
def leer_jpg(iid):
    return np.asarray(Image.open(io.BytesIO(ZIP.read(f"images/{iid}.jpg"))).convert("RGB"))
print("[+] Dataset y split v1 verificados")

OUT = DRIVE_ROOT / "results" / "m1" / "etapa6"; OUT.mkdir(parents=True, exist_ok=True)
FIG = DRIVE_ROOT / "figures" / "m1" / "etapa6"; FIG.mkdir(parents=True, exist_ok=True)
CODE_DRIVE = DRIVE_ROOT / "code" / "m1"; CODE_DRIVE.mkdir(parents=True, exist_ok=True)

## 2. Módulo `src/augmentation.py`
La celda escribe el archivo en la carpeta del repo (`%%writefile`) y la siguiente lo importa. En la Etapa 7 irá al repositorio junto con `make_dataset()`, en un PR propio.

In [ ]:
%%writefile src/augmentation.py
"""Aumento de datos del Módulo 1 (Etapa 6). Solo se aplica en entrenamiento.

Entrada: imagen float32 (H, W, 3) en [0, 255], ya recortada y redimensionada
(después de la caché del pipeline). Salida: misma forma y rango.

Todas las operaciones son "stateless": dado el mismo `seed` (tensor int64 de 2 valores)
la imagen aumentada es siempre la misma. El pipeline (Etapa 7) genera un seed distinto
por imagen y época, así que cada época ve variantes nuevas y el resultado es reproducible.

Dos intensidades (aumento selectivo):
  - fuerte (todas las clases salvo nv): volteos, rotación, zoom, brillo y contraste.
  - ligero (nv): solo volteos horizontal y vertical, que no pierden ni inventan píxeles.
"""
import math
import tensorflow as tf

CLASES = ["akiec", "bcc", "bkl", "df", "mel", "nv", "vasc"]

# Parámetros por defecto = los de configs/base.yaml (data.augmentation).
# Revisión visual (315 variantes, 35 lesiones, revisor no dermatólogo): oscurecimiento leve en el 5,6 %
# sin pérdida de los rasgos de la lesión; el zoom toca la lesión en el 0,7 %. Se mantienen los valores.
CONFIG_POR_DEFECTO = {
    "apply_to": ["akiec", "bcc", "bkl", "df", "mel", "vasc"],   # intensidad fuerte
    "horizontal_flip": True,
    "vertical_flip": True,
    "rotation_factor": 0.25,    # fracción de vuelta: ±0,25 × 360° = ±90°. Con los dos volteos cubre los 360°
    "zoom_factor": 0.1,         # solo acercar: el lado visible pasa a ser 90–100 % (≈ 81–100 % del área)
    "brightness_delta": 0.1,    # brillo multiplicativo ×[0,9, 1,1]
    "contrast_factor": 0.1,     # contraste ×[0,9, 1,1] respecto a la media de cada canal
    "fill_mode": "REFLECT",     # relleno al rotar: reflejo (nunca negro: fabricaría un borde de dermatoscopio)
}

def _subseeds(seed, n):
    return tf.unstack(tf.random.experimental.stateless_split(tf.cast(seed, tf.int64), num=n))

def _voltear(img, seed, cfg):
    s1, s2 = _subseeds(seed, 2)
    if cfg["horizontal_flip"]:
        img = tf.image.stateless_random_flip_left_right(img, s1)
    if cfg["vertical_flip"]:
        img = tf.image.stateless_random_flip_up_down(img, s2)
    return img

def rotar(img, angulo, fill_mode="REFLECT"):
    """Rota `angulo` radianes alrededor del centro; las esquinas se rellenan según fill_mode."""
    h = tf.cast(tf.shape(img)[0], tf.float32)
    w = tf.cast(tf.shape(img)[1], tf.float32)
    c, s = tf.cos(angulo), tf.sin(angulo)
    cx, cy = (w - 1) / 2, (h - 1) / 2
    # Transformación inversa (salida -> entrada), formato de ImageProjectiveTransformV3
    t = tf.stack([c, -s, cx - c * cx + s * cy, s, c, cy - s * cx - c * cy, 0.0, 0.0])
    out = tf.raw_ops.ImageProjectiveTransformV3(
        images=img[tf.newaxis], transforms=t[tf.newaxis], output_shape=tf.shape(img)[:2],
        fill_value=0.0, interpolation="BILINEAR", fill_mode=fill_mode)
    return out[0]

def _fuerte(img, seed, cfg):
    s_flip, s_rot, s_zoom, s_crop, s_bri, s_con = _subseeds(seed, 6)
    img = _voltear(img, s_flip, cfg)
    # Rotación continua
    if cfg["rotation_factor"] > 0:
        lim = cfg["rotation_factor"] * 2 * math.pi
        ang = tf.random.stateless_uniform([], s_rot, -lim, lim)
        img = rotar(img, ang, cfg["fill_mode"])
    # Zoom (solo acercar): recorte aleatorio de lado 90–100 % y vuelta al tamaño original
    if cfg["zoom_factor"] > 0:
        h, w = tf.shape(img)[0], tf.shape(img)[1]
        frac = tf.random.stateless_uniform([], s_zoom, 1.0 - cfg["zoom_factor"], 1.0)
        lado_h = tf.cast(tf.round(frac * tf.cast(h, tf.float32)), tf.int32)
        lado_w = tf.cast(tf.round(frac * tf.cast(w, tf.float32)), tf.int32)
        img = tf.image.stateless_random_crop(img, tf.stack([lado_h, lado_w, 3]), s_crop)
        img = tf.image.resize(img, tf.stack([h, w]), method="bilinear")
    # Brillo y contraste (no se toca tono ni saturación: el color es criterio diagnóstico)
    if cfg["brightness_delta"] > 0:
        d = cfg["brightness_delta"]
        img = img * tf.random.stateless_uniform([], s_bri, 1.0 - d, 1.0 + d)
    if cfg["contrast_factor"] > 0:
        f = cfg["contrast_factor"]
        img = tf.image.stateless_random_contrast(img, 1.0 - f, 1.0 + f, s_con)
    return tf.clip_by_value(img, 0.0, 255.0)

def _ligero(img, seed, cfg):
    return _voltear(img, seed, cfg)

def augmentar(img, etiqueta, seed, cfg=None):
    """Aumenta UNA imagen según su clase. `etiqueta` = índice de clase (orden de CLASES)."""
    cfg = {**CONFIG_POR_DEFECTO, **(cfg or {})}
    fuertes = tf.constant([CLASES.index(c) for c in cfg["apply_to"]], dtype=tf.int32)
    es_fuerte = tf.reduce_any(tf.equal(tf.cast(etiqueta, tf.int32), fuertes))
    img = tf.cast(img, tf.float32)
    return tf.cond(es_fuerte, lambda: _fuerte(img, seed, cfg), lambda: _ligero(img, seed, cfg))

def preparar(img, etiqueta, seed, training, cfg=None):
    """Punto de entrada del pipeline: aumenta solo si training=True; si no, devuelve la imagen tal cual."""
    if training:
        return augmentar(img, etiqueta, seed, cfg)
    return tf.cast(img, tf.float32)

In [ ]:
import importlib, src.augmentation as A
importlib.reload(A)
shutil.copy("src/augmentation.py", CODE_DRIVE / "augmentation.py")      # copia persistente en Drive

# Los parámetros deben coincidir con configs/base.yaml (lo acordado por el grupo)
aug_yaml = cfg["data"]["augmentation"]
for k, v in aug_yaml.items():
    esperado = CAMBIOS_PENDIENTES_YAML.get(k, v)
    assert A.CONFIG_POR_DEFECTO[k] == esperado, f"[!] {k}: se esperaba {esperado}, el módulo tiene {A.CONFIG_POR_DEFECTO[k]}"
print("[+] Parámetros iguales a base.yaml" + (", salvo los cambios pendientes de PR:" if CAMBIOS_PENDIENTES_YAML else ":"), aug_yaml)
for k, v in CAMBIOS_PENDIENTES_YAML.items():
    print(f"   {k}: base.yaml {aug_yaml.get(k)} → módulo {v}")
print("Añadido por el Módulo 1 (no está en base.yaml): fill_mode =", A.CONFIG_POR_DEFECTO["fill_mode"],
      "· intensidad ligera (volteos) para nv")

## 3. Catálogo de transformaciones
| Transformación | Rango | Decisión | Clases | Justificación clínica |
|---|---|---|---|---|
| Volteo horizontal y vertical | aleatorio | Usar | Todas | Una lesión no tiene orientación; el diagnóstico no cambia al reflejarla |
| Rotación | ±90° (con los volteos cubre los 360°) | Usar | Todas salvo nv | En dermatoscopia no hay "arriba" |
| Relleno al rotar | reflejo | Usar | — | El relleno negro fabricaría un borde de dermatoscopio, presente en el 42 % de mel frente al 6 % de nv (Etapa 3) |
| Zoom (solo acercar) | lado 90–100 % (≈ 81–100 % del área) | Usar | Todas salvo nv | Simula distancias de captura; acercar poco evita cortar el borde de la lesión (criterio B) |
| Brillo | ×0,9–1,1 | Usar | Todas salvo nv | Simula diferencias de iluminación entre equipos. Revisión visual: oscurecimiento leve en el 5,6 % de las variantes, sin pérdida de los rasgos de la lesión |
| Contraste | ×0,9–1,1 | Usar | Todas salvo nv | Igual que el brillo |
| Tono y saturación | — | No usar | — | El color es criterio diagnóstico (C de ABCDE); el problema de color entre equipos (Etapa 3) se estudia con constancia de color en la ablación |
| Alejar (zoom out) | — | No usar | — | Obliga a rellenar los bordes con píxeles inventados |
| Deformación elástica | — | No usar | — | Puede alterar la simetría (criterio A) |
| Borrado de zonas (cutout) | — | No usar | — | Puede tapar la parte diagnóstica de la lesión |

**Por qué nv solo con volteos:** es el 67 % de train; los volteos no pierden ni inventan píxeles y le dan algo de variedad, pero el aumento fuerte se reserva para las clases con pocas imágenes. Comparar *sin aumento · uniforme · selectivo* es el experimento 2 de la ablación.

In [ ]:
catalogo = pd.DataFrame([
    ("volteo_horizontal_vertical", "aleatorio", "usar", "todas", "la lesión no tiene orientación"),
    ("rotacion", "±90° (+ volteos = 360°)", "usar", "todas salvo nv", "en dermatoscopia no hay arriba"),
    ("relleno_rotacion", "reflejo", "usar", "-", "el negro fabricaría un borde de dermatoscopio (42 % mel vs 6 % nv)"),
    ("zoom_acercar", "lado 90-100 %", "usar", "todas salvo nv", "distancias de captura sin cortar el borde (B)"),
    ("brillo", "x0,9-1,1", "usar", "todas salvo nv", "iluminación; revisión visual: oscurecimiento leve en el 5,6 % sin pérdida de rasgos"),
    ("contraste", "x0,9-1,1", "usar", "todas salvo nv", "iluminación"),
    ("tono_saturacion", "-", "no usar", "-", "el color es criterio diagnóstico (C)"),
    ("zoom_alejar", "-", "no usar", "-", "rellena con píxeles inventados"),
    ("deformacion_elastica", "-", "no usar", "-", "altera la simetría (A)"),
    ("cutout", "-", "no usar", "-", "puede tapar la zona diagnóstica"),
], columns=["transformacion", "rango", "decision", "clases", "justificacion"])
catalogo.to_csv(OUT / "catalogo_aumento.csv", index=False)
print("[+] Catálogo guardado en", OUT / "catalogo_aumento.csv")

## 4. Preprocesado previo (provisional hasta la Etapa 7)
Recorte central 600×450 → 450×450 y reducción a 224×224 por área. Es lo que recibirá el aumento dentro del pipeline.

In [ ]:
def a_224(arr):
    h, w = arr.shape[:2]; s = min(h, w)
    arr = arr[(h - s) // 2:(h + s) // 2, (w - s) // 2:(w + s) // 2]
    return tf.image.resize(tf.constant(arr, tf.float32), [224, 224], method="area")

## 5. Comprobaciones automáticas

In [ ]:
CL = A.CLASES
tr = split[split.split == "train"]
iid = tr[tr.dx == "mel"].sample(1, random_state=0).image_id.iloc[0]
img = a_224(leer_jpg(iid))
s = tf.constant([7, 11], tf.int64)

a1, a2 = A.augmentar(img, CL.index("mel"), s), A.augmentar(img, CL.index("mel"), s)
assert bool(tf.reduce_all(a1 == a2)), "[!] No es reproducible"
print("[+] Reproducible: el mismo seed da la misma variante")
assert not bool(tf.reduce_all(A.augmentar(img, CL.index("mel"), tf.constant([8, 11], tf.int64)) == a1))
print("[+] Otro seed da otra variante")
for c in CL:
    assert bool(tf.reduce_all(A.preparar(img, CL.index(c), s, training=False) == img))
print("[+] Con training=False la imagen sale intacta en las 7 clases (validación y prueba sin aumento)")
assert a1.shape == img.shape and float(tf.reduce_min(a1)) >= 0 and float(tf.reduce_max(a1)) <= 255
print("[+] Misma forma y rango [0, 255]")
b = A.augmentar(img, CL.index("nv"), s).numpy()
assert np.allclose(np.sort(b.ravel()), np.sort(img.numpy().ravel()))
print("[+] nv: solo volteos (mismos píxeles, solo reordenados)")

## 6. Figura de variantes por clase (revisión visual)
Una imagen de train por clase: el original y 9 variantes. **Revisar con calma** y anotar si alguna variante:
- deja de parecer la misma lesión,
- corta el borde de la lesión (zoom excesivo),
- cambia el color de forma visible,
- muestra esquinas raras por la rotación.

Si se desea revisar más casos, cambia `SEMILLA_FIG` en la sección 1 y volver a ejecutar esta celda.

In [ ]:
N_VAR = 9
ejemplos = tr.groupby("dx").sample(1, random_state=SEMILLA_FIG).set_index("dx").reindex(CL)
import matplotlib.pyplot as plt
fig, axes = plt.subplots(len(CL), N_VAR + 1, figsize=(1.25 * (N_VAR + 1), 1.3 * len(CL)))
for r, c in enumerate(CL):
    base = a_224(leer_jpg(ejemplos.loc[c, "image_id"]))
    variantes = [base] + [A.augmentar(base, CL.index(c), tf.constant([SEMILLA_FIG, 100 * r + j], tf.int64))
                          for j in range(N_VAR)]
    for j, v in enumerate(variantes):
        ax = axes[r, j]; ax.imshow(v.numpy().astype("uint8")); ax.set_xticks([]); ax.set_yticks([])
        for sp in ax.spines.values():
            sp.set_visible(j == 0); sp.set_color("#1f1f1d"); sp.set_linewidth(1.5)
    axes[r, 0].set_ylabel(f"{c}\n({'ligero' if c == 'nv' else 'fuerte'})", rotation=0, ha="right",
                          va="center", fontsize=9, labelpad=6)
axes[0, 0].set_title("original", fontsize=9)
fig.subplots_adjust(wspace=0.04, hspace=0.06)
fig.savefig(FIG / f"fig_variantes_aumento_s{SEMILLA_FIG}.png", dpi=300, bbox_inches="tight", facecolor="white")
plt.show()
print("Imágenes usadas:", ejemplos.image_id.to_dict())

## 7. Figura: relleno negro frente a reflejo
La misma imagen rotada 30°. Con relleno negro aparecen esquinas negras idénticas a la viñeta del dermatoscopio, el artefacto que en la Etapa 3 resultó siete veces más frecuente en melanoma que en nevus.

In [ ]:
base = a_224(leer_jpg(ejemplos.loc["mel", "image_id"]))
ang = tf.constant(np.deg2rad(30), tf.float32)
paneles = [(base, "Original"), (A.rotar(base, ang, "CONSTANT"), "Rotación 30°, relleno negro"),
           (A.rotar(base, ang, "REFLECT"), "Rotación 30°, relleno por reflejo")]
fig, axes = plt.subplots(1, 3, figsize=(9, 3.2))
for ax, (im_, t) in zip(axes, paneles):
    ax.imshow(im_.numpy().astype("uint8")); ax.set_title(t, fontsize=9); ax.axis("off")
fig.tight_layout()
fig.savefig(FIG / "fig_relleno_negro_vs_reflejo.png", dpi=300, bbox_inches="tight", facecolor="white")
plt.show()
negro = lambda x: 100 * float(tf.reduce_mean(tf.cast(tf.reduce_max(x, -1) < 5, tf.float32)))
print(f"Píxeles negros tras rotar 30°: relleno negro {negro(paneles[1][0]):.1f} % · reflejo {negro(paneles[2][0]):.1f} %")

## 8. Registro en MLflow

In [ ]:
import mlflow
from src.tracking import setup_mlflow, standard_tags
os.environ["MLFLOW_TRACKING_USERNAME"] = userdata.get("DAGSHUB_USER")
os.environ["MLFLOW_TRACKING_PASSWORD"] = userdata.get("DAGSHUB_TOKEN")
setup_mlflow(cfg); mlflow.set_experiment(cfg["mlflow"]["experiments"]["m1"])
shutil.copy(SPLIT_DRIVE, cfg["paths"]["split_csv"])
tags = standard_tags(cfg, modulo="m1", fase="aumento", autor=AUTOR, dataset_sha256=DATASET_SHA256_ESPERADO)
assert tags["split_sha256"] == SPLIT_SHA256_ESPERADO
# Si ya hubo un run m1-aumento con otros parámetros (p. ej. brillo 0,1), queda como descartado
for r in mlflow.search_runs(filter_string="attributes.run_name = 'm1-aumento'", output_format="list"):
    if r.data.params.get("aug_brightness_delta") != str(A.CONFIG_POR_DEFECTO["brightness_delta"]):
        mlflow.MlflowClient().set_tag(r.info.run_id, "estado", "descartado")
        mlflow.MlflowClient().set_tag(r.info.run_id, "motivo", "parámetros de aumento distintos a los finales")
        print("Run anterior marcado como descartado:", r.info.run_id)
with mlflow.start_run(run_name="m1-aumento", tags=tags) as run:
    p = {f"aug_{k}": (",".join(v) if isinstance(v, list) else v) for k, v in A.CONFIG_POR_DEFECTO.items()}
    mlflow.log_params({**p, "aug_modo": "selectivo", "aug_nv": "volteos", "aug_en_linea": True,
                       "preproc_previo": "recorte_central_450+area_224 (provisional)",
                       "revision_visual": "315 variantes, 35 lesiones, revisor no dermatologo; oscurecimiento leve 5,6 % sin perdida de rasgos; zoom toca la lesion 0,7 %; se mantienen los valores"})
    mlflow.log_artifact("src/augmentation.py", artifact_path="codigo")
    mlflow.log_artifacts(str(OUT), artifact_path="etapa6")
    mlflow.log_artifacts(str(FIG), artifact_path="figuras")
print("[+] Run m1-aumento registrado:", run.info.run_id)